In [14]:
edges = [
    ("A", "B", "red", 1, "cable"),  
    ("B", "C", "blue", 3, "trolley"), 
    ("C", "D", "green", 4, "trolley"),  
    ("D", "I", "green", 9, "bus"),  
    ("I", "J", "blue", 14, "horse"),
    ("J", "N", "red", 23, "bus"),
    ("C", "G", "red", 7, "cable"),
    ("H", "G", "red", 12, "cable"),
    ("I", "G", "red", 13, "horse"),
    ("H", "L", "red", 20, "bus"),
    ("R", "L", "red", 29, "horse"),
    ("R", "x", "red", 40, "trolley"),
    ("c", "x", "red", 53, "horse"),
    ("c", "h", "red", 64, "cable"),
    ("h", "i", "red", 69, "bus"),
    ("d", "i", "red", 65, "horse"),
    ("e", "j", "red", 66, "bus"),
    ("d", "S", "red", 42, "horse"),
    ("M", "S", "red", 30, "trolley"),
    ("N", "T", "red", 31, "bus"),
    ("D", "J", "red", 5, "trolley"),
    ("E", "O", "red", 15, "cable"),
    ("u", "O", "red", 36, "bus"),
    ("B", "F", "red", 6, "trolley"),
    ("K", "F", "red", 17, "horse"),
    ("K", "P", "red", 27, "cable"),
    ("Q", "W", "red", 39, "trolley"),
    ("V", "W", "red", 46, "cable"),
    ("b", "W", "red", 52, "trolley"),
    ("b", "g", "red", 62, "trolley"),
    ("b", "a", "red", 56, "cable"),
    ("z", "a", "red", 55, "trolley"),
    ("z", "V", "red", 49, "cable"),

    ("B", "E", "blue", 2, "cable"),
    ("D", "G", "blue", 8, "bus"),
    ("K", "G", "blue", 18, "bus"),
    ("K", "O", "blue", 26, "trolley"),
    ("V", "O", "blue", 37, "horse"),
    ("V", "b", "blue", 51, "trolley"),
    ("h", "b", "blue", 63, "bus"),
    ("f", "g", "blue", 67, "cable"),
    ("f", "z", "blue", 60, "horse"),
    ("u", "z", "blue", 48, "bus"),
    ("i", "j", "blue", 70, "bus"),
    ("M", "N", "blue", 25, "horse"),
    ("M", "L", "blue", 24, "bus"),
    ("T", "S", "blue", 35, "bus"),
    ("R", "S", "blue", 34, "horse"),
    ("R", "Q", "blue", 33, "cable"),

    ("E", "F", "green", 10, "horse"),
    ("F", "G", "green", 11, "horse"),
    ("E", "K", "green", 16, "bus"),
    ("I", "N", "green", 22, "horse"),
    ("I", "M", "green", 21, "trolley"),
    ("G", "L", "green", 19, "trolley"),
    ("L", "Q", "green", 28, "trolley"),
    ("P", "Q", "green", 32, "cable"),
    ("V", "P", "green", 38, "horse"),
    ("V", "U", "green", 45, "bus"),
    ("V", "a", "green", 50, "bus"),
    ("f", "a", "green", 61, "horse"),
    ("g", "h", "green", 68, "cable"),
    ("b", "c", "green", 57, "trolley"),
    ("c", "d", "green", 58, "bus"),
    ("e", "d", "green", 59, "horse"),
    ("e", "y", "green", 54, "trolley"),
    ("S", "y", "green", 43, "cable"),
    ("T", "y", "green", 44, "trolley"),
    ("S", "x", "green", 41, "bus"),
    ("W", "x", "green", 47, "cable"),
]

In [5]:
import networkx as nx

# ایجاد گراف از داده‌ها

# ساخت گراف
G = nx.Graph()
for u, v, color, weight, line_type in edges:
    G.add_edge(u, v, color=color, weight=weight, line_type=line_type)

def find_path_with_one_fare(G, start, end):
    """
    پیدا کردن مسیری که از قوانین انتقال رایگان تبعیت کند و از یک کرایه استفاده کند.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS برای پیدا کردن مسیر
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    while queue:
        node, path, last_edge = queue.pop(0)
        if node == end:
            return path

        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None

# پیدا کردن مسیر از A به j
path = find_path_with_one_fare(G, "A", "j")
if path:
    print("Path with one fare:", " -> ".join(path))
else:
    print("No path found.")



Path with one fare: A -> B -> F -> K -> P -> Q -> R -> S -> d -> i -> h -> i -> j


In [11]:
def find_path_with_one_fare_fixed(G, start, end):
    """
    پیدا کردن مسیری که از قوانین انتقال رایگان تبعیت کند، از یک کرایه استفاده کند،
    و مسیر بهینه بدون بازدیدهای تکراری باشد.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با ردیابی گره‌های بازدید شده
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    visited = set()  # گره‌های بازدید شده

    while queue:
        node, path, last_edge = queue.pop(0)
        if node in visited:
            continue  # اگر گره قبلاً بازدید شده، ادامه بده

        visited.add(node)  # علامت‌گذاری گره به‌عنوان بازدیدشده

        if node == end:
            return path

        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None

# اجرای نسخه بهینه‌شده
optimized_path = find_path_with_one_fare_fixed(G, "A", "j")
if optimized_path:
    print("Optimized Path with one fare:", " -> ".join(optimized_path))
else:
    print("No path found.")


No path found.


In [6]:
def find_path_with_distance_constraint(G, start, end):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. هر گره حداقل با فاصله دو گره دیگر دوباره دیده شود.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با محدودیت فاصله بازدید مجدد از گره‌ها
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    visited = {}  # نگه‌داری گره‌ها و تعداد گره‌هایی که از آخرین بازدید آن‌ها گذشته است

    while queue:
        node, path, last_edge = queue.pop(0)

        # به‌روزرسانی فاصله بازدید گره‌ها
        for visited_node in visited:
            visited[visited_node] += 1

        # بررسی اینکه گره فعلی اخیراً بازدید نشده باشد
        if node in visited and visited[node] < 2:
            continue

        visited[node] = 0  # بازنشانی فاصله گره فعلی

        if node == end:
            return path

        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None

# اجرای نسخه اصلاح‌شده
distance_constrained_path = find_path_with_distance_constraint(G, "A", "j")
if distance_constrained_path:
    print("Path with distance constraint:", " -> ".join(distance_constrained_path))
else:
    print("No path found.")


Path with distance constraint: A -> B -> F -> K -> P -> Q -> R -> S -> d -> i -> h -> i -> j


In [35]:
def find_path_with_distance_constraint(G, start, end):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. هر گره حداقل با فاصله دو گره دیگر دوباره دیده شود.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با محدودیت فاصله بازدید مجدد از گره‌ها
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    visited = {}  # نگه‌داری آخرین زمان بازدید هر گره

    while queue:
        node, path, last_edge = queue.pop(0)

        # به‌روزرسانی فاصله بازدید گره‌ها
        for visited_node in visited:
            visited[visited_node] += 1

        # بررسی اینکه گره فعلی اخیراً بازدید نشده باشد
        if node in visited and visited[node] < 2:
            continue

        visited[node] = 0  # بازنشانی فاصله گره فعلی

        if node == end:
            return path

        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                if neighbor not in path or visited[neighbor] >= 3:  # بررسی فاصله
                    queue.append((neighbor, path + [neighbor], edge_data))

    return None

# اجرای نسخه اصلاح‌شده
distance_constrained_path = find_path_with_distance_constraint(G, "A", "j")
if distance_constrained_path:
    print("Path with distance constraint:", " -> ".join(distance_constrained_path))
else:
    print("No path found.")


Path with distance constraint: A -> B -> F -> K -> P -> Q -> R -> S -> d -> i -> h -> i -> j


In [10]:
def find_path_without_backtracking(G, start, end):
    """
    پیدا کردن مسیری از start به end:
    1. بدون بازگشت به رأس قبلی.
    2. حرکت فقط رو به جلو (مهم نیست یال‌ها تکراری باشند).
    """
    from collections import deque

    # BFS برای جستجوی مسیر
    queue = deque([(start, [start])])  # (گره فعلی، مسیر طی شده)
    visited_edges = set()  # یال‌های بازدیدشده برای اطمینان از حرکت رو به جلو

    while queue:
        node, path = queue.popleft()

        # اگر به مقصد رسیدیم، مسیر را برگردانیم
        if node == end:
            return path

        # بررسی تمام همسایه‌های گره فعلی
        for neighbor in G.neighbors(node):
            edge = (node, neighbor)

            # اطمینان از اینکه بازگشتی به گره قبلی وجود ندارد
            if len(path) > 1 and neighbor == path[-2]:
                continue

            # بررسی و اضافه‌کردن یال به مسیر
            if edge not in visited_edges:
                visited_edges.add(edge)
                queue.append((neighbor, path + [neighbor]))

    return None  # اگر مسیری یافت نشد
# اجرای نسخه اصلاح‌شده
distance_constrained_path = find_path_with_distance_constraint(G, "A", "j")
if distance_constrained_path:
    print("Path with distance constraint:", " -> ".join(distance_constrained_path))
else:
    print("No path found.")

TypeError: unhashable type: 'dict'

In [11]:
def find_path_with_distance_constraint(G, start, end):
    """
    پیدا کردن مسیری از start به end:
    1. قوانین انتقال رایگان را رعایت کند.
    2. هر گره حداقل با فاصله دو گره دیگر دوباره دیده شود.
    3. مسیر بدون بازگشت باشد.
    """
    from collections import deque

    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS برای جستجو
    queue = deque([(start, [start], None)])  # (گره فعلی، مسیر، اطلاعات آخرین یال)
    visited = {}  # نگه‌داری گره‌ها و فاصله آن‌ها از آخرین بازدید

    while queue:
        node, path, last_edge = queue.popleft()

        # به‌روزرسانی فاصله بازدید گره‌ها
        visited = {key: value + 1 for key, value in visited.items()}

        # بررسی اینکه گره فعلی اخیراً بازدید نشده باشد
        if (node, last_edge) in visited and visited[(node, last_edge)] < 2:
            continue

        visited[(node, last_edge)] = 0  # بازنشانی فاصله گره فعلی

        # اگر به مقصد رسیدیم، مسیر را برگردانیم
        if node == end:
            return path

        # بررسی تمام همسایه‌های گره فعلی
        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]
            edge_key = (edge_data["color"], edge_data["line_type"])  # تبدیل یال به قابل هش
            if last_edge is None or can_transfer(last_edge, edge_key):
                queue.append((neighbor, path + [neighbor], edge_key))

    return None  # اگر مسیری یافت نشد


# اجرای نسخه اصلاح‌شده
distance_constrained_path = find_path_with_distance_constraint(G, "A", "j")
if distance_constrained_path:
    print("Path with distance constraint:", " -> ".join(distance_constrained_path))
else:
    print("No path found.")

TypeError: tuple indices must be integers or slices, not str

In [13]:
import networkx as nx

def find_path_with_strict_distance_constraint(G, start, end):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. حداقل دو گره بین بازدید مجدد از یک گره وجود داشته باشد.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با ردیابی مسیرها و رعایت فاصله دو گره
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)

    while queue:
        node, path, last_edge = queue.pop(0)

        # بررسی شرط پایان
        if node == end:
            return path

        for neighbor in G.neighbors(node):
            if neighbor in path[-2:]:
                continue  # اگر گره در دو گره آخر مسیر باشد، از آن صرف‌نظر کن

            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None

# ایجاد گراف
G = nx.Graph()

# افزودن یال‌ها
edges = [
    ("A", "B", "red", 1, "cable"),  
    ("B", "C", "blue", 3, "trolley"), 
    ("C", "D", "green", 4, "trolley"),  
    ("D", "I", "green", 9, "bus"),  
    ("I", "J", "blue", 14, "horse"),
    ("J", "N", "red", 23, "bus"),
    ("C", "G", "red", 7, "cable"),
    ("H", "G", "red", 12, "cable"),
    ("I", "G", "red", 13, "horse"),
    ("H", "L", "red", 20, "bus"),
    ("R", "L", "red", 29, "horse"),
    ("R", "x", "red", 40, "trolley"),
    ("c", "x", "red", 53, "horse"),
    ("c", "h", "red", 64, "cable"),
    ("h", "i", "red", 69, "bus"),
    ("d", "i", "red", 65, "horse"),
    ("e", "j", "red", 66, "bus"),
    ("d", "S", "red", 42, "horse"),
    ("M", "S", "red", 30, "trolley"),
    ("N", "T", "red", 31, "bus"),
    ("D", "J", "red", 5, "trolley"),
    ("E", "O", "red", 15, "cable"),
    ("u", "O", "red", 36, "bus"),
    ("B", "F", "red", 6, "trolley"),
    ("K", "F", "red", 17, "horse"),
    ("K", "P", "red", 27, "cable"),
    ("Q", "W", "red", 39, "trolley"),
    ("V", "W", "red", 46, "cable"),
    ("b", "W", "red", 52, "trolley"),
    ("b", "g", "red", 62, "trolley"),
    ("b", "a", "red", 56, "cable"),
    ("z", "a", "red", 55, "trolley"),
    ("z", "V", "red", 49, "cable"),

    ("B", "E", "blue", 2, "cable"),
    ("D", "G", "blue", 8, "bus"),
    ("K", "G", "blue", 18, "bus"),
    ("K", "O", "blue", 26, "trolley"),
    ("V", "O", "blue", 37, "horse"),
    ("V", "b", "blue", 51, "trolley"),
    ("h", "b", "blue", 63, "bus"),
    ("f", "g", "blue", 67, "cable"),
    ("f", "z", "blue", 60, "horse"),
    ("u", "z", "blue", 48, "bus"),
    ("i", "j", "blue", 70, "bus"),
    ("M", "N", "blue", 25, "horse"),
    ("M", "L", "blue", 24, "bus"),
    ("T", "S", "blue", 35, "bus"),
    ("R", "S", "blue", 34, "horse"),
    ("R", "Q", "blue", 33, "cable"),

    ("E", "F", "green", 10, "horse"),
    ("F", "G", "green", 11, "horse"),
    ("E", "K", "green", 16, "bus"),
    ("I", "N", "green", 22, "horse"),
    ("I", "M", "green", 21, "trolley"),
    ("G", "L", "green", 19, "trolley"),
    ("L", "Q", "green", 28, "trolley"),
    ("P", "Q", "green", 32, "cable"),
    ("V", "P", "green", 38, "horse"),
    ("V", "U", "green", 45, "bus"),
    ("V", "a", "green", 50, "bus"),
    ("f", "a", "green", 61, "horse"),
    ("g", "h", "green", 68, "cable"),
    ("b", "c", "green", 57, "trolley"),
    ("c", "d", "green", 58, "bus"),
    ("e", "d", "green", 59, "horse"),
    ("e", "y", "green", 54, "trolley"),
    ("S", "y", "green", 43, "cable"),
    ("T", "y", "green", 44, "trolley"),
    ("S", "x", "green", 41, "bus"),
    ("W", "x", "green", 47, "cable"),
]

# اضافه کردن یال‌ها به گراف
for u, v, color, weight, line_type in edges:
    G.add_edge(u, v, color=color, weight=weight, line_type=line_type)

# اجرای کد
start_node = "A"
end_node = "j"
strict_distance_path = find_path_with_strict_distance_constraint(G, start_node, end_node)

# نمایش خروجی
if strict_distance_path:
    print("Path with strict distance constraint:", " -> ".join(strict_distance_path))
else:
    print("No path found.")


KeyboardInterrupt: 

In [4]:
def find_path_with_strict_distance_constraint(G, start, end):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. حداقل دو گره بین بازدید مجدد از یک گره وجود داشته باشد.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با ردیابی مسیرها و رعایت فاصله دو گره
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)

    while queue:
        node, path, last_edge = queue.pop(0)

        # بررسی شرط پایان
        if node == end:
            return path

        for neighbor in G.neighbors(node):
            if neighbor in path[-2:]:
                continue  # اگر گره در دو گره آخر مسیر باشد، از آن صرف‌نظر کن

            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None

# اجرای نسخه بهینه‌شده با محدودیت فاصله
strict_distance_path = find_path_with_strict_distance_constraint(G, "A", "j")
if strict_distance_path:
    print("Path with strict distance constraint:", " -> ".join(strict_distance_path))
else:
    print("No path found.")


KeyboardInterrupt: 

In [5]:
def find_path_with_strict_distance_constraint_fixed(G, start, end, max_depth=1000):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. حداقل دو گره بین بازدید مجدد از یک گره وجود داشته باشد.
    4. محدودیت عمق برای جلوگیری از حلقه بی‌پایان اعمال شود.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با محدودیت عمق
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    visited = set()  # نگه‌داری مسیرهای بازدید شده

    while queue:
        node, path, last_edge = queue.pop(0)

        # بررسی عمق مسیر
        if len(path) > max_depth:
            continue  # اگر مسیر از عمق مجاز بیشتر شد، آن را نادیده بگیر

        # بررسی شرط پایان
        if node == end:
            return path

        for neighbor in G.neighbors(node):
            if neighbor in path[-2:]:
                continue  # اگر گره در دو گره آخر مسیر باشد، از آن صرف‌نظر کن

            edge_data = G[node][neighbor]
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], edge_data))

    return None


In [7]:
def find_path_no_backtracking(G, start, end):
    """
    پیدا کردن مسیری که:
    1. قوانین انتقال رایگان را رعایت کند.
    2. از یک کرایه استفاده کند.
    3. از یالی که وارد شده‌ایم، نمی‌توانیم برگردیم.
    """
    def can_transfer(edge1, edge2):
        # بررسی قوانین انتقال رایگان
        return edge1["color"] == edge2["color"] or edge1["line_type"] == edge2["line_type"]

    # BFS با شرط عدم بازگشت از یال
    queue = [(start, [start], None)]  # (گره فعلی، مسیر، یال قبلی)
    visited_edges = set()  # نگه‌داری یال‌هایی که استفاده شده‌اند

    while queue:
        node, path, last_edge = queue.pop(0)

        # بررسی شرط پایان
        if node == end:
            return path

        for neighbor in G.neighbors(node):
            edge_data = G[node][neighbor]

            # شرط بازگشت از یال
            if last_edge is not None and (node, neighbor) == (last_edge[1], last_edge[0]):
                continue  # اگر یال بازگشتی باشد، آن را نادیده بگیر

            # بررسی قوانین انتقال رایگان
            if last_edge is None or can_transfer(last_edge, edge_data):
                queue.append((neighbor, path + [neighbor], (node, neighbor)))
                visited_edges.add((node, neighbor))

    return None


In [8]:
# اجرای کد
strict_distance_path = find_path_no_backtracking(G, "A", "j")

# نمایش خروجی
if strict_distance_path:
    print("Path without backtracking:", " -> ".join(strict_distance_path))
else:
    print("No path found.")


TypeError: tuple indices must be integers or slices, not str

In [6]:
strict_distance_path = find_path_with_strict_distance_constraint_fixed(G, "A", "j", max_depth=1000)

if strict_distance_path:
    print("Path with strict distance constraint:", " -> ".join(strict_distance_path))
else:
    print("No path found.")


KeyboardInterrupt: 

In [16]:
import networkx as nx


# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع برای قوانین انتقال رایگان
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"] or  # انتقال رایگان اگر رنگ یکسان باشد
        prev_edge["line_type"] == current_edge["line_type"]  # یا نوع خط یکسان باشد
    )

# الگوریتم Dijkstra با قوانین انتقال رایگان
def find_optimal_route(graph, start, end):
    queue = [(0, start, None)]  # (هزینه، گره فعلی، یال قبلی)
    visited = set()
    paths = {start: [start]}  # مسیرهای ردیابی شده

    while queue:
        cost, node, prev_edge = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)

        # اگر به گره پایان برسیم، مسیر را برگردانیم
        if node == end:
            return cost, paths[node]

        # بررسی همسایگان
        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            if neighbor not in visited and is_valid_transition(prev_edge, edge_data):
                total_cost = cost + edge_data["weight"]
                queue.append((total_cost, neighbor, edge_data))
                queue.sort()  # مرتب‌سازی برای الگوریتم Dijkstra
                paths[neighbor] = paths[node] + [neighbor]

    return float("inf"), []  # اگر مسیری پیدا نشود

# پیدا کردن مسیر بهینه از A به g
start_node = "A"
end_node = "j"
cost, path = find_optimal_route(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Minimum cost: {cost}")
    print(f"Optimal Path: {' -> '.join(path)}")
else:
    print("No valid path found.")


No valid path found.


In [17]:
import networkx as nx


# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع برای قوانین انتقال رایگان
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"] or  # انتقال رایگان اگر رنگ یکسان باشد
        prev_edge["line_type"] == current_edge["line_type"]  # یا نوع خط یکسان باشد
    )

# الگوریتم BFS با قوانین انتقال رایگان
def find_route_with_bfs(graph, start, end):
    queue = [(start, None, [start])]  # (گره فعلی، یال قبلی، مسیر فعلی)
    visited = set()

    while queue:
        node, prev_edge, path = queue.pop(0)

        # اگر گره پایان پیدا شد، مسیر را برگردانید
        if node == end:
            return path

        visited.add(node)

        # بررسی همسایگان
        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            if neighbor not in visited and is_valid_transition(prev_edge, edge_data):
                queue.append((neighbor, edge_data, path + [neighbor]))

    return None  # اگر مسیری پیدا نشود

# پیدا کردن مسیر ممکن از A به g
start_node = "A"
end_node = "j"
path = find_route_with_bfs(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Valid Path: {' -> '.join(path)}")
else:
    print("No valid path found.")


No valid path found.


In [18]:
import networkx as nx

# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع برای قوانین انتقال رایگان
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"]  # انتقال رایگان اگر رنگ یکسان باشد
        or prev_edge["line_type"] == current_edge["line_type"]  # انتقال رایگان اگر نوع خط یکسان باشد
    )

# الگوریتم BFS با قوانین انتقال رایگان
def find_route_with_bfs(graph, start, end):
    queue = [(start, None, [start], 0)]  # (گره فعلی، یال قبلی، مسیر فعلی، هزینه فعلی)
    visited = set()

    while queue:
        node, prev_edge, path, cost = queue.pop(0)

        # اگر گره پایان پیدا شد، مسیر و هزینه را برگردانید
        if node == end:
            return path, cost

        visited.add((node, prev_edge["color"] if prev_edge else None, prev_edge["line_type"] if prev_edge else None))

        # بررسی همسایگان
        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            edge_color = edge_data["color"]
            edge_type = edge_data["line_type"]
            if (neighbor, edge_color, edge_type) not in visited and is_valid_transition(prev_edge, edge_data):
                new_cost = cost + (0 if prev_edge and (prev_edge["color"] == edge_color or prev_edge["line_type"] == edge_type) else edge_data["weight"])
                queue.append((neighbor, edge_data, path + [neighbor], new_cost))

    return None, None  # اگر مسیری پیدا نشود

# پیدا کردن مسیر ممکن از A به j
start_node = "A"
end_node = "j"
path, cost = find_route_with_bfs(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Valid Path: {' -> '.join(path)}")
    print(f"Total Cost: {cost}")
else:
    print("No valid path found.")


Valid Path: A -> B -> F -> K -> P -> Q -> R -> S -> d -> i -> h -> i -> j
Total Cost: 1


In [19]:
import networkx as nx

# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف (گراف بدون جهت)
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع برای قوانین انتقال رایگان
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"]  # انتقال رایگان اگر رنگ یکسان باشد
        or prev_edge["line_type"] == current_edge["line_type"]  # انتقال رایگان اگر نوع خط یکسان باشد
    )

# الگوریتم BFS با قوانین انتقال رایگان
def find_route_with_bfs(graph, start, end):
    queue = [(start, None, [start], 0)]  # (گره فعلی، یال قبلی، مسیر فعلی، هزینه فعلی)
    visited = set()

    while queue:
        node, prev_edge, path, cost = queue.pop(0)

        # اگر گره پایان پیدا شد، مسیر و هزینه را برگردانید
        if node == end:
            return path, cost

        visited.add((node, prev_edge["color"] if prev_edge else None, prev_edge["line_type"] if prev_edge else None))

        # بررسی همسایگان
        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            edge_color = edge_data["color"]
            edge_type = edge_data["line_type"]
            if (neighbor, edge_color, edge_type) not in visited and is_valid_transition(prev_edge, edge_data):
                new_cost = cost + (0 if prev_edge and (prev_edge["color"] == edge_color or prev_edge["line_type"] == edge_type) else edge_data["weight"])
                queue.append((neighbor, edge_data, path + [neighbor], new_cost))

    return None, None  # اگر مسیری پیدا نشود

# پیدا کردن مسیر ممکن از A به j
start_node = "A"
end_node = "j"
path, cost = find_route_with_bfs(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Valid Path: {' -> '.join(path)}")
    print(f"Total Cost: {cost}")
else:
    print("No valid path found.")


Valid Path: A -> B -> F -> K -> P -> Q -> R -> S -> d -> i -> h -> i -> j
Total Cost: 1


In [20]:
import networkx as nx

# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف (گراف بدون جهت)
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع برای قوانین انتقال رایگان
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"]  # انتقال رایگان اگر رنگ یکسان باشد
        or prev_edge["line_type"] == current_edge["line_type"]  # انتقال رایگان اگر نوع خط یکسان باشد
    )

# الگوریتم BFS با جلوگیری از بازدید مجدد از یال
def find_route_with_bfs(graph, start, end):
    queue = [(start, None, [start], 0, set())]  # (گره فعلی، یال قبلی، مسیر فعلی، هزینه فعلی، یال‌های بازدیدشده)
    visited_edges = set()

    while queue:
        node, prev_edge, path, cost, used_edges = queue.pop(0)

        # اگر گره پایان پیدا شد، مسیر و هزینه را برگردانید
        if node == end:
            return path, cost

        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            edge = tuple(sorted((node, neighbor)))  # یال بدون جهت

            # اگر این یال قبلاً بازدید نشده باشد و انتقال معتبر باشد
            if edge not in used_edges and is_valid_transition(prev_edge, edge_data):
                new_cost = cost + (0 if prev_edge and (prev_edge["color"] == edge_data["color"] or prev_edge["line_type"] == edge_data["line_type"]) else edge_data["weight"])
                new_used_edges = used_edges.copy()
                new_used_edges.add(edge)
                queue.append((neighbor, edge_data, path + [neighbor], new_cost, new_used_edges))

    return None, None  # اگر مسیری پیدا نشود

# پیدا کردن مسیر ممکن از A به j
start_node = "A"
end_node = "j"
path, cost = find_route_with_bfs(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Valid Path: {' -> '.join(path)}")
    print(f"Total Cost: {cost}")
else:
    print("No valid path found.")


No valid path found.


In [21]:
import networkx as nx
import heapq

# ساخت گراف
graph = nx.Graph()

# افزودن یال‌ها به گراف (گراف بدون جهت)
for edge in edges:
    graph.add_edge(edge[0], edge[1], color=edge[2], weight=edge[3], line_type=edge[4])

# تابع بررسی انتقال معتبر
def is_valid_transition(prev_edge, current_edge):
    if prev_edge is None:
        return True  # اولین یال معتبر است
    return (
        prev_edge["color"] == current_edge["color"]  # انتقال رایگان اگر رنگ یکسان باشد
        or prev_edge["line_type"] == current_edge["line_type"]  # انتقال رایگان اگر نوع خط یکسان باشد
    )

# الگوریتم Dijkstra با جلوگیری از بازگشت مستقیم
def find_route(graph, start, end):
    pq = []  # صف اولویت‌دار
    heapq.heappush(pq, (0, start, None, [start]))  # (هزینه فعلی، گره فعلی، یال قبلی، مسیر فعلی)
    visited = set()

    while pq:
        cost, node, prev_edge, path = heapq.heappop(pq)

        # اگر به مقصد رسیدیم، مسیر و هزینه را برگردانید
        if node == end:
            return path, cost

        # جلوگیری از بازدید مستقیم یال‌های قبلی
        visited.add((node, prev_edge["edge"] if prev_edge else None))

        # بررسی همسایگان
        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            edge = (node, neighbor)
            reverse_edge = (neighbor, node)

            # جلوگیری از بازگشت مستقیم به یال قبلی
            if prev_edge and reverse_edge == prev_edge["edge"]:
                continue

            # اگر انتقال معتبر باشد
            if (neighbor, edge) not in visited:
                new_cost = cost + (0 if prev_edge and is_valid_transition(prev_edge, edge_data) else edge_data["weight"])
                heapq.heappush(pq, (new_cost, neighbor, {"edge": edge, **edge_data}, path + [neighbor]))

    return None, None  # اگر مسیری پیدا نشود

# پیدا کردن مسیر از A به j
start_node = "A"
end_node = "j"
path, cost = find_route(graph, start_node, end_node)

# چاپ نتیجه
if path:
    print(f"Valid Path: {' -> '.join(path)}")
    print(f"Total Cost: {cost}")
else:
    print("No valid path found.")


TypeError: '<' not supported between instances of 'dict' and 'dict'